# LLM EKG: Detecting Silent Behavioral Tampering

**The problem**: API providers can silently alter an LLM's behavior — by changing the
system prompt, restricting output length, or swapping the model entirely. You're paying
for full Grok-3 capabilities, but behind the scenes the provider adds constraints to
cut costs (shorter responses = fewer output tokens billed). No notification. No changelog.

**The solution**: [LLM EKG](https://pypi.org/project/llm-ekg/) monitors 16 statistical
features of LLM responses in real-time. No NLP. No ML. Pure mathematics.
When the behavioral signature shifts, the EKG detects it — even when the model is the same.

## What this notebook demonstrates

1. **Experiment 1 (Mid-session tampering)**: 25 prompts through normal Grok-3, then 25
   prompts through Grok-3 **with a hidden system prompt that restricts output**. Same
   model, same prompts — but the EKG detects the behavioral shift immediately.

2. **Experiment 2 (Security baseline check)**: Save the normal Grok-3 fingerprint as a
   security baseline. Run the tampered Grok-3 through a new monitor and compare.
   Status: **COMPROMISED** with 5+ features beyond 3-sigma threshold.

**Cost**: < $0.10 total | **Time**: ~5 minutes | **Dependencies**: `llm-ekg`, `openai`

In [1]:
# ============================================================
# CONFIGURATION  (change these values)
# ============================================================
import os

XAI_API_KEY = os.environ.get("XAI_API_KEY", "xai-YOUR-KEY-HERE")  # or paste directly
XAI_BASE_URL = "https://api.x.ai/v1"

MODEL = "grok-3"  # The model being monitored

# The hidden system prompt that the provider silently injects.
# This simulates a provider trying to reduce output token costs.
TAMPERED_SYSTEM_PROMPT = """You are a concise assistant. Follow these rules strictly:
1. Never use more than 3 sentences total.
2. Never use bullet points, numbered lists, or headers.
3. Never use bold or italic formatting.
4. Just give a brief, plain text answer."""

MAX_TOKENS = 300
DELAY = 0.3  # seconds between API calls (rate limiting)

In [2]:
# ============================================================
# SETUP: imports, prompts, helpers
# ============================================================
# !pip install llm-ekg openai matplotlib numpy  # uncomment if needed

import time
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend (remove if running in Jupyter)
import matplotlib.pyplot as plt
from openai import OpenAI
from llm_ekg import LiveMonitor, SecurityBaseline, FEATURE_NAMES

# --- 25 homogeneous analytical prompts ---
# All follow the same format: "Compare X and Y. Give pros and cons."
# Homogeneous prompts = low within-model variance = high detection sensitivity.

PROMPTS = [
    "Compare solar and wind energy for residential homes. Give pros and cons.",
    "Compare electric and gasoline cars for daily commuting. Give pros and cons.",
    "Compare cloud and on-premise servers for small businesses. Give pros and cons.",
    "Compare remote and office work for software developers. Give pros and cons.",
    "Compare Python and JavaScript for web development. Give pros and cons.",
    "Compare renting and buying a home in a major city. Give pros and cons.",
    "Compare public and private schools for education quality. Give pros and cons.",
    "Compare freelancing and full-time employment for a programmer. Give pros and cons.",
    "Compare SQL and NoSQL databases for a startup. Give pros and cons.",
    "Compare iOS and Android for mobile app development. Give pros and cons.",
    "Compare agile and waterfall methodologies for project management. Give pros and cons.",
    "Compare nuclear and solar energy for large-scale power generation. Give pros and cons.",
    "Compare online and in-person learning for university students. Give pros and cons.",
    "Compare Docker and virtual machines for application deployment. Give pros and cons.",
    "Compare TypeScript and plain JavaScript for large codebases. Give pros and cons.",
    "Compare SSDs and HDDs for data storage in servers. Give pros and cons.",
    "Compare React and Vue.js for building web applications. Give pros and cons.",
    "Compare public transport and personal cars for city commuting. Give pros and cons.",
    "Compare microservices and monolithic architecture. Give pros and cons.",
    "Compare fiber optic and cable internet for home use. Give pros and cons.",
    "Compare machine learning and rule-based systems for fraud detection. Give pros and cons.",
    "Compare open-source and proprietary software for enterprise use. Give pros and cons.",
    "Compare 5G and WiFi 6 for smart home connectivity. Give pros and cons.",
    "Compare Kubernetes and Docker Compose for container orchestration. Give pros and cons.",
    "Compare REST and GraphQL for API design. Give pros and cons.",
]

print(f"Prompts: {len(PROMPTS)}")
print(f"Total API calls: {len(PROMPTS) * 3} (baseline + tampered mid-session + tampered check)")

# --- Helper ---

def call_batch(client, prompts, monitor, phase_name, system_prompt=None):
    """Call API for each prompt and display real-time EKG metrics."""
    results = []
    total = len(prompts)
    for i, prompt in enumerate(prompts, 1):
        try:
            t0 = time.time()
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": prompt})
            response = client.chat.completions.create(
                model=MODEL, messages=messages,
                max_tokens=MAX_TOKENS, temperature=0.7,
            )
            elapsed = time.time() - t0
            text = response.choices[0].message.content or ""

            monitor.ingest(text, response_time_s=elapsed, model=MODEL)

            n = len(monitor.analyzer.state_history)
            if n > 0:
                latest = monitor.analyzer.state_history[-1]
                anom = latest["anomaly_score"]
                drift = latest["drift"]
                icon = "G" if anom < 0.3 else ("Y" if anom < 0.6 else "R")
                print(f"  [{icon}] {phase_name} {i:2d}/{total} | "
                      f"anomaly={anom:.3f} drift={drift:.4f} | "
                      f"{elapsed:.1f}s | {len(text):4d} chars | {text[:50]}...")
            results.append(text)
        except Exception as e:
            print(f"  [!] {phase_name} {i}/{total} ERROR: {e}")
            time.sleep(2)
        time.sleep(DELAY)
    return results

Prompts: 25
Total API calls: 75 (baseline + tampered mid-session + tampered check)


In [3]:
# ============================================================
# EXPERIMENT 1: Mid-Session Behavioral Tampering
# ============================================================
# Single monitor. First 25 prompts = normal Grok-3.
# Then 25 prompts = SAME Grok-3, but with hidden system prompt.
# Same model, same prompts — only the system prompt changed.

client = OpenAI(api_key=XAI_API_KEY, base_url=XAI_BASE_URL)
monitor = LiveMonitor()

# --- Phase A: Normal Grok-3 ---
print("=" * 70)
print(f"PHASE A: Normal {MODEL} - {len(PROMPTS)} prompts")
print("=" * 70)

call_batch(client, PROMPTS, monitor, "NORM")

summary_before = monitor.summary()
swap_point = len(monitor.analyzer.state_history)
print(f"\nBaseline complete: {summary_before['verdict']} "
      f"({summary_before['global_score_100']}/100)")
print(f"Mean anomaly: {summary_before['global_anomaly_mean']:.4f}")

# Save security baseline
baseline = SecurityBaseline.from_analyzer(monitor.analyzer, model=MODEL)
baseline.save("baseline_grok3_normal.json")
print(f"Security baseline saved ({swap_point} responses)")

# --- Phase B: Tampered Grok-3 (hidden system prompt) ---
print(f"\n{'=' * 70}")
print(f"PHASE B: TAMPERED {MODEL} - hidden system prompt restricts output")
print(f"Same model, same prompts — provider silently injects constraints")
print("=" * 70)

call_batch(client, PROMPTS, monitor, "TAMP", system_prompt=TAMPERED_SYSTEM_PROMPT)

summary_after = monitor.summary()
print(f"\nAfter tampering: {summary_after['verdict']} "
      f"({summary_after['global_score_100']}/100)")
print(f"Mean anomaly: {summary_after['global_anomaly_mean']:.4f}")
print(f"Trend: {summary_after['trend']:+.4f} "
      f"({'RISING' if summary_after['trend'] > 0.01 else 'STABLE'})")

# --- Timeline plot ---
anomalies = [h["anomaly_score"] for h in monitor.analyzer.state_history]
drifts = [h["drift"] for h in monitor.analyzer.state_history]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle("Experiment 1: Mid-Session Behavioral Tampering Detection",
             fontsize=14, fontweight="bold")

ax1.plot(range(swap_point), anomalies[:swap_point], color="#22c55e", linewidth=2,
         label=f"Normal {MODEL}")
ax1.plot(range(swap_point, len(anomalies)), anomalies[swap_point:], color="#ef4444",
         linewidth=2, label=f"Tampered {MODEL}")
ax1.axvline(x=swap_point, color="#eab308", linestyle="--", alpha=0.8,
            label="Tampering starts")
ax1.axhline(y=0.5, color="gray", linestyle=":", alpha=0.4)
ax1.set_ylabel("Anomaly Score")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.2)
ax1.set_ylim(-0.05, 1.05)

ax2.bar(range(swap_point), drifts[:swap_point], color="#22c55e", alpha=0.6, width=1.0)
ax2.bar(range(swap_point, len(drifts)), drifts[swap_point:], color="#ef4444",
        alpha=0.6, width=1.0)
ax2.axvline(x=swap_point, color="#eab308", linestyle="--", alpha=0.8)
ax2.set_ylabel("State Drift")
ax2.set_xlabel("Response Index")
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("exp1_tampering_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp1_tampering_timeline.png")

PHASE A: Normal grok-3 - 25 prompts


  [G] NORM  1/25 | anomaly=0.000 drift=0.0000 | 18.3s | 1565 chars | Solar and wind energy are two of the most popular ...


  [G] NORM  2/25 | anomaly=0.000 drift=0.1964 | 5.5s | 1515 chars | When comparing electric cars (EVs) and gasoline ca...


  [G] NORM  3/25 | anomaly=0.000 drift=0.3120 | 4.3s | 1627 chars | When deciding between cloud and on-premise servers...


  [G] NORM  4/25 | anomaly=0.000 drift=0.4168 | 4.1s | 1573 chars | Remote and office work for software developers eac...


...

  [G] TAMP 24/25 | anomaly=0.200 drift=0.0326 | 5.3s |  555 chars | Kubernetes is a powerful, scalable container orche...


  [G] TAMP 25/25 | anomaly=0.151 drift=0.0282 | 3.6s |  550 chars | REST is a widely-used architectural style for APIs...



After tampering: DEGRADED (78/100)
Mean anomaly: 0.1520
Trend: +0.2124 (RISING)


Saved: exp1_tampering_timeline.png


/tmp/ipykernel_369695/2087135892.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 1: Analysis

The timeline above shows the behavioral EKG of a single monitoring session.

**Before tampering** (green): The anomaly score stays low and stable. Normal Grok-3
produces consistent, detailed responses — ~1500 chars each, with headers, bullet
points, and structured formatting.

**After tampering** (red): The anomaly score and drift spike immediately. The
hidden system prompt forces the model to produce short, plain-text responses
(~500 chars). The same model, answering the same questions, now has a
completely different behavioral fingerprint.

**Key insight**: LLM EKG doesn't need to see the system prompt or know the model
name. It detects behavioral changes purely from the statistical properties of the
output text — length, vocabulary diversity, formatting patterns, assertion density.

In [4]:
# ============================================================
# EXPERIMENT 2: Security Baseline Check
# ============================================================
# New monitor with ONLY tampered responses.
# Compare against the normal Grok-3 baseline.

print("=" * 70)
print(f"EXPERIMENT 2: Security Baseline Check")
print(f"Running tampered {MODEL} and comparing against normal baseline")
print("=" * 70)

monitor_check = LiveMonitor()
call_batch(client, PROMPTS, monitor_check, "CHECK",
           system_prompt=TAMPERED_SYSTEM_PROMPT)

# Security check
sec_report = baseline.check(monitor_check.analyzer, sigma=3.0)

status_colors = {"CLEAN": "32", "WARNING": "33", "COMPROMISED": "31"}
c = status_colors.get(sec_report.status, "0")
print(f"\n{'=' * 70}")
print(f"SECURITY STATUS: \033[{c}m{sec_report.status}\033[0m")
print(f"Deviated features: {sec_report.n_deviated}/{len(sec_report.deviations)}")
print(f"Threshold: {sec_report.sigma_threshold} sigma")
print(f"{'=' * 70}")

# Show deviations
deviated = [d for d in sec_report.deviations if d["deviated"]]
deviated.sort(key=lambda x: x["weighted_z"], reverse=True)

if deviated:
    print(f"\nFeatures that EXCEEDED the 3-sigma threshold:")
    for d in deviated:
        direction = "+" if d["current_mean"] > d["baseline_mean"] else "-"
        pct = abs(d["current_mean"] - d["baseline_mean"]) / (d["baseline_mean"] + 1e-10) * 100
        print(f"  {d['name']:25s} z={d['weighted_z']:6.2f}  "
              f"(normal={d['baseline_mean']:.2f} -> tampered={d['current_mean']:.2f} "
              f"[{direction}{pct:.0f}%])")
else:
    print("\nNo significant deviations detected.")

# All features table
print(f"\nAll 16 features (sorted by deviation):")
all_sorted = sorted(sec_report.deviations, key=lambda x: x["weighted_z"], reverse=True)
for d in all_sorted:
    marker = " !!" if d["deviated"] else "   "
    print(f"{marker} {d['name']:25s} z={d['weighted_z']:6.2f}  "
          f"normal={d['baseline_mean']:8.2f}  tampered={d['current_mean']:8.2f}")

# --- Deviation bar chart ---
fig, ax = plt.subplots(figsize=(10, 7))
names = [d["name"] for d in all_sorted]
z_scores = [d["weighted_z"] for d in all_sorted]
colors = ["#ef4444" if d["deviated"] else "#22c55e" for d in all_sorted]

y_pos = np.arange(len(names))
bars = ax.barh(y_pos, z_scores, color=colors)
ax.axvline(x=sec_report.sigma_threshold, color="#eab308", linestyle="--",
           linewidth=2, label=f"{sec_report.sigma_threshold}$\\sigma$ threshold")
ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Weighted Z-Score")
ax.set_title(f"Security Check: Tampered vs Normal {MODEL}\n"
             f"Status: {sec_report.status} | "
             f"{sec_report.n_deviated}/16 features deviated",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="x", alpha=0.2)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("exp2_security_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp2_security_check.png")

EXPERIMENT 2: Security Baseline Check
Running tampered grok-3 and comparing against normal baseline


  [G] CHECK  1/25 | anomaly=0.000 drift=0.0000 | 11.0s |  482 chars | Solar energy for homes is reliable in sunny areas,...


  [G] CHECK  2/25 | anomaly=0.000 drift=0.1976 | 9.3s |  373 chars | Electric cars are ideal for daily commuting due to...


  [G] CHECK  3/25 | anomaly=0.000 drift=0.3128 | 11.3s |  441 chars | Cloud servers offer small businesses scalability a...


  [G] CHECK  4/25 | anomaly=0.000 drift=0.4167 | 1.2s |  434 chars | Remote work for software developers offers flexibi...


...

  [G] CHECK 24/25 | anomaly=0.130 drift=0.4881 | 2.2s |  539 chars | Kubernetes is a powerful, scalable container orche...


  [G] CHECK 25/25 | anomaly=0.232 drift=0.0554 | 2.0s |  600 chars | REST is a widely-used architectural style for APIs...



SECURITY STATUS: COMPROMISED
Deviated features: 6/16
Threshold: 3.0 sigma

Features that EXCEEDED the 3-sigma threshold:
  length_words              z= 15.58  (normal=219.40 -> tampered=68.88 [-69%])
  length_chars              z= 11.25  (normal=1542.52 -> tampered=472.96 [-69%])
  sentence_count            z=  4.76  (normal=16.12 -> tampered=2.84 [-82%])
  avg_sentence_length       z=  4.33  (normal=14.02 -> tampered=24.62 [+76%])
  vocab_diversity           z=  3.88  (normal=0.66 -> tampered=0.81 [+24%])
  list_ratio                z=  3.69  (normal=0.56 -> tampered=0.00 [-100%])

All 16 features (sorted by deviation):
 !! length_words              z= 15.58  normal=  219.40  tampered=   68.88
 !! length_chars              z= 11.25  normal= 1542.52  tampered=  472.96
 !! sentence_count            z=  4.76  normal=   16.12  tampered=    2.84
 !! avg_sentence_length       z=  4.33  normal=   14.02  tampered=   24.62
 !! vocab_diversity           z=  3.88  normal=    0.66  tampered=    

Saved: exp2_security_check.png


/tmp/ipykernel_369695/3880942963.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ============================================================
# COMPARATIVE VISUALIZATION
# ============================================================

fh_normal = np.array(monitor.analyzer.feature_history[:swap_point])
fh_tampered = np.array(monitor_check.analyzer.feature_history)

key_features = ["length_words", "vocab_diversity", "sentence_count",
                "avg_sentence_length", "list_ratio", "assertion_density"]
key_indices = [FEATURE_NAMES.index(f) for f in key_features]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f"Feature Comparison: Normal vs Tampered {MODEL}",
             fontsize=14, fontweight="bold")

for ax, fname, fidx in zip(axes.flat, key_features, key_indices):
    normal_vals = fh_normal[:, fidx]
    tampered_vals = fh_tampered[:, fidx]

    bp = ax.boxplot(
        [normal_vals, tampered_vals],
        tick_labels=["Normal", "Tampered"],
        patch_artist=True, widths=0.5,
    )
    bp["boxes"][0].set_facecolor("#22c55e")
    bp["boxes"][0].set_alpha(0.6)
    bp["boxes"][1].set_facecolor("#ef4444")
    bp["boxes"][1].set_alpha(0.6)

    ax.set_title(fname, fontsize=10, fontweight="bold")
    ax.grid(axis="y", alpha=0.2)

    bl_mean = normal_vals.mean()
    sw_mean = tampered_vals.mean()
    pct = ((sw_mean - bl_mean) / (bl_mean + 1e-10)) * 100
    ax.set_xlabel(f"mean: {bl_mean:.1f} vs {sw_mean:.1f} ({pct:+.0f}%)", fontsize=8)

plt.tight_layout()
plt.savefig("exp3_feature_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp3_feature_comparison.png")

# --- Summary table ---
s_normal = summary_before
s_tampered = monitor_check.summary()

bl_anomalies = [h['anomaly_score'] for h in monitor.analyzer.state_history[:swap_point]]
bl_anomaly_mean = np.mean(bl_anomalies) if bl_anomalies else 0

print(f"\n{'=' * 70}")
print(f"{'Metric':<30} {'Normal':>20} {'Tampered':>20}")
print(f"{'-' * 70}")
print(f"{'Responses':<30} {swap_point:>20} {s_tampered['n_responses']:>20}")
print(f"{'Avg response length (chars)':<30} {fh_normal[:, 0].mean():>20.0f} {fh_tampered[:, 0].mean():>20.0f}")
print(f"{'Global Score':<30} {s_normal['global_score_100']:>19}/100 {s_tampered['global_score_100']:>19}/100")
print(f"{'Verdict':<30} {s_normal['verdict']:>20} {s_tampered['verdict']:>20}")
print(f"{'Mean Anomaly':<30} {bl_anomaly_mean:>20.4f} {s_tampered['global_anomaly_mean']:>20.4f}")
print(f"{'Security Status':<30} {'BASELINE':>20} {sec_report.status:>20}")
print(f"{'Features Deviated (3-sigma)':<30} {'n/a':>20} {sec_report.n_deviated:>18}/16")
print(f"{'=' * 70}")

Saved: exp3_feature_comparison.png

Metric                                       Normal             Tampered
----------------------------------------------------------------------
Responses                                        25                   25
Avg response length (chars)                    1543                  473
Global Score                                    85/100                  88/100
Verdict                                     HEALTHY              HEALTHY
Mean Anomaly                                 0.0458               0.0355
Security Status                            BASELINE          COMPROMISED
Features Deviated (3-sigma)                     n/a                  6/16


/tmp/ipykernel_369695/477836411.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ============================================================
# GENERATE HTML REPORTS
# ============================================================

monitor.report("report_tampering_detected.html", security_report=sec_report)
monitor_check.report("report_tampered_check.html", security_report=sec_report)

print("HTML reports generated (open in browser for full EKG visualization):")
print("  1. report_tampering_detected.html - Full session with tampering detection")
print("  2. report_tampered_check.html     - Standalone security check")
print("\nPNG charts generated:")
print("  1. exp1_tampering_timeline.png    - Anomaly timeline")
print("  2. exp2_security_check.png        - Security deviation bar chart")
print("  3. exp3_feature_comparison.png    - Feature box plots")

Report generated: report_tampering_detected.html (50 responses)


Report generated: report_tampered_check.html (25 responses)
HTML reports generated (open in browser for full EKG visualization):
  1. report_tampering_detected.html - Full session with tampering detection
  2. report_tampered_check.html     - Standalone security check

PNG charts generated:
  1. exp1_tampering_timeline.png    - Anomaly timeline
  2. exp2_security_check.png        - Security deviation bar chart
  3. exp3_feature_comparison.png    - Feature box plots


## Conclusions

### What LLM EKG detected

1. **Real-time anomaly spike**: The moment the provider injected a hidden system prompt,
   the behavioral fingerprint shifted dramatically. Response length dropped ~70%, list
   formatting disappeared, vocabulary diversity changed.

2. **Security baseline violation**: The security check flagged **COMPROMISED** with 5+
   features beyond the 3-sigma threshold. Z-scores >10 on length metrics.

3. **Feature-level forensics**: The security report identifies *which* features changed
   and by how much, enabling root cause analysis of the behavioral shift.

### Why this matters

This demo used the **same model** for both normal and tampered responses. The only
difference was a hidden system prompt. This demonstrates that LLM EKG detects
**any behavioral change**, not just model swaps:

- **System prompt injection/tampering**
- **Silent model downgrades** (GPT-4o to GPT-3.5-turbo)
- **Parameter changes** (temperature, max_tokens)
- **Fine-tuning drift** (model updates that change behavior)

### Design note: homogeneous prompts

This demo uses 25 prompts of the same type ("Compare X and Y"). This is intentional:
homogeneous prompts produce low within-model variance, which maximizes detection
sensitivity. In production, you should create baselines per prompt category for
best results.

### Use in production

```python
from openai import OpenAI
from llm_ekg import LiveMonitor

monitor = LiveMonitor()
client = monitor.wrap_openai(OpenAI())  # transparent wrapper

# Use client normally — monitoring is automatic
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "..."}]
)

# Check health at any time
print(f"Score: {monitor.score}/100, Verdict: {monitor.verdict}")

# Save baseline and detect future changes
monitor.save_baseline("my_baseline.json", model="gpt-4o")
report = monitor.security_check("my_baseline.json")
if report.status != "CLEAN":
    alert_ops_team(f"Model behavior changed: {report.status}")
```

### Install

```bash
pip install llm-ekg
```

GitHub: [github.com/iafiscal1212/llm-ekg](https://github.com/iafiscal1212/llm-ekg)  
PyPI: [pypi.org/project/llm-ekg](https://pypi.org/project/llm-ekg/)